In [0]:
# Enable inference logging — done already via auto_capture_config in Step 6
# Inference table: fda_rag.gold.fda_rag_inference_payload

# Build operational metrics view
spark.sql("""
CREATE OR REPLACE VIEW fda_rag.gold.v_inference_metrics AS
SELECT
  date_trunc('hour', from_unixtime(timestamp_ms/1000)) AS hour,
  COUNT(*) AS request_count,
  AVG(execution_time_ms) AS avg_latency_ms,
  PERCENTILE(execution_time_ms, 0.50) AS p50_latency_ms,
  PERCENTILE(execution_time_ms, 0.95) AS p95_latency_ms,
  SUM(CASE WHEN status_code >= 400 THEN 1 ELSE 0 END) AS errors,
  SUM(CASE WHEN response LIKE '%don''t have enough%' THEN 1 ELSE 0 END) AS refusals
FROM fda_rag.gold.fda_rag_inference_payload
GROUP BY 1
""")

display(spark.sql("SELECT * FROM fda_rag.gold.v_inference_metrics ORDER BY hour DESC LIMIT 24"))